In [ ]:

!nvidia-smi

# Install required packages
!pip install -q transformers==4.41.0 datasets torch mlflow==3.1.4 boto3 scikit-learn google-cloud-storage dvc[gs] peft==0.11.1 accelerate==0.30.1

Tue Feb 24 07:56:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!unzip processed.zip

Archive:  processed.zip
   creating: processed/
  inflating: processed/dataset_dict.json  
   creating: processed/train/
  inflating: processed/train/data-00000-of-00001.arrow  
  inflating: processed/train/state.json  
  inflating: processed/train/dataset_info.json  
   creating: processed/validation/
  inflating: processed/validation/data-00000-of-00001.arrow  
  inflating: processed/validation/state.json  
  inflating: processed/validation/dataset_info.json  
   creating: processed/test/
  inflating: processed/test/data-00000-of-00001.arrow  
  inflating: processed/test/state.json  
  inflating: processed/test/dataset_info.json  
  inflating: processed/label_config.json  


In [ ]:
from datasets import load_from_disk
import json
from transformers import (
  AutoModelForSequenceClassification,
  AutoTokenizer,
  Trainer,
  TrainingArguments,
  pipeline,
)
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset = load_from_disk("/content/processed")
with open('/content/processed/label_config.json', 'r') as f:
    label_config = json.load(f)
print(f"\n🏷️  Number of labels: {label_config['num_labels']}")
print(f"   Entity types: {len(label_config['label_names'])} labels")



🏷️  Number of labels: 21
   Entity types: 21 labels


In [ ]:
import mlflow

In [ ]:
MLFLOW_TRACKING_URI = "http://34.143.169.73:5000/"

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [ ]:
mlflow.set_experiment("ner")

<Experiment: artifact_location='s3://mlflow/artifacts/1', creation_time=1771918540015, experiment_id='1', last_update_time=1771918540015, lifecycle_stage='active', name='ner', tags={}>

In [ ]:
client = mlflow.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
experiments = client.search_experiments()
print(f"\n✅ MLflow connection successful!")
print(f"   Total experiments: {len(experiments)}")



✅ MLflow connection successful!
   Total experiments: 2


In [ ]:
experiment = mlflow.get_experiment_by_name("ner")
experiment_id = experiment.experiment_id
print(f"name :{experiment.name}")
print(f"id :{experiment_id}")

name :ner
id :1


In [ ]:
sample = dataset['train'][0]
print(f"   Input IDs length: {len(sample['input_ids'])}")
print(f"   Labels length: {len(sample['labels'])}")
print(f"   First 10 input IDs: {sample['input_ids'][:10]}")
print(f"   First 10 labels: {sample['labels'][:10]}")

   Input IDs length: 200
   Labels length: 200
   First 10 input IDs: [0, 2316, 790, 4, 326, 2142, 917, 9170, 2927, 380]
   First 10 labels: [-100, 20, 20, 20, 20, 20, 20, 20, 20, 20]


In [ ]:
TRAINING_CONFIG = {
    'model_name': 'vinai/phobert-base-v2',
    'num_labels': label_config['num_labels'],
    'epochs': 4,
    'batch_size': 16,
    'learning_rate': 5e-5,
    'weight_decay': 0.01,
    'warmup_steps': 500,
    'max_grad_norm': 1.0,
    'logging_steps': 100,
    'eval_steps': 500,
    'save_steps': 500,
    'output_dir': 'models/phobert-ner',
}

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TRAINING_CONFIG['model_name'], use_fast=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
id2label = {int(k): v for k, v in label_config['id_to_label'].items()}

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    TRAINING_CONFIG['model_name'],
    num_labels=TRAINING_CONFIG['num_labels'],
    id2label=id2label,
    label2id=label_config['label_to_id'],
)

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

RobertaForTokenClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

In [ ]:
from sklearn.metrics import classification_report, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    """
    Compute metrics for NER evaluation
    Ignores -100 labels (padding/special tokens)
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_labels = []
    true_predictions = []

    for prediction, label in zip(predictions, labels):
        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:  # Ignore padding
                true_labels.append(label_id)
                true_predictions.append(pred_id)

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        average='weighted',
        zero_division=0
    )

    # Calculate per-class metrics
    precision_per_class, recall_per_class, f1_per_class, support = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        average=None,
        zero_division=0
    )

    # Calculate accuracy
    accuracy = np.mean(np.array(true_predictions) == np.array(true_labels))

    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

    # Add per-class F1 for important classes (entities)
    id_to_label = label_config['id_to_label']
    for idx, (p, r, f) in enumerate(zip(precision_per_class, recall_per_class, f1_per_class)):
        label = id_to_label.get(str(idx), f"label_{idx}")
        if label != 'O' and label.startswith('B-'):  # Only B- tags
            entity_type = label.split('-')[1]
            metrics[f'f1_{entity_type}'] = f

    return metrics

In [ ]:
from transformers import TrainingArguments, DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("✅ Data collator created")

# Training arguments
training_args = TrainingArguments(
    output_dir=TRAINING_CONFIG['output_dir'],

    # Training schedule
    num_train_epochs=TRAINING_CONFIG['epochs'],
    per_device_train_batch_size=TRAINING_CONFIG['batch_size'],
    per_device_eval_batch_size=TRAINING_CONFIG['batch_size'],

    # Optimization
    learning_rate=TRAINING_CONFIG['learning_rate'],
    weight_decay=TRAINING_CONFIG['weight_decay'],
    warmup_steps=TRAINING_CONFIG['warmup_steps'],
    max_grad_norm=TRAINING_CONFIG['max_grad_norm'],

    # Evaluation and logging
    eval_strategy='steps',
    eval_steps=TRAINING_CONFIG['eval_steps'],
    logging_steps=TRAINING_CONFIG['logging_steps'],
    logging_dir=f"{TRAINING_CONFIG['output_dir']}/logs",

    # Saving
    save_strategy='steps',
    save_steps=TRAINING_CONFIG['save_steps'],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,

    # Performance
    fp16=True,  # Mixed precision training on GPU
    dataloader_num_workers=2,

    # Misc
    report_to='mlflow',  # We use MLflow for logging
    push_to_hub=False,
    disable_tqdm=False,
)


✅ Data collator created


In [ ]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
mlflow.set_experiment("ner")

<Experiment: artifact_location='s3://mlflow/artifacts/1', creation_time=1771918540015, experiment_id='1', last_update_time=1771918540015, lifecycle_stage='active', name='ner', tags={}>

In [ ]:

with mlflow.start_run() as run:
    trainer.train()


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Age,F1 Date,F1 Gender,F1 Job,F1 Location,F1 Name,F1 Organization,F1 Patient Id,F1 Symptom And Disease,F1 Transportation
500,0.165900,0.165018,0.970613,0.966107,0.970613,0.968174,0.930900,0.988631,0.869707,0.000000,0.951717,0.922252,0.879061,0.975439,0.898166,0.982456
1000,0.056700,0.084606,0.981356,0.981272,0.981356,0.981299,0.975477,0.988278,0.948097,0.785185,0.969764,0.938338,0.926961,0.985133,0.916935,1.000000


🏃 View run rogue-bass-551 at: http://34.143.169.73:5000/#/experiments/1/runs/5ca4ee601d234786ae84a93e41384918
🧪 View experiment at: http://34.143.169.73:5000/#/experiments/1


In [ ]:
import os
import torch
# import pipeline

In [ ]:
os.environ['MLFLOW_S3_ENDPOINT_URL'] = "http://35.240.144.213:9000" # Cổng MinIO bạn đã sửa ở bước trước
os.environ['AWS_ACCESS_KEY_ID'] = "minioadmin"                # Mặc định của MinIO
os.environ['AWS_SECRET_ACCESS_KEY'] = "minioadmin123"

In [ ]:
tuned_pipeline = pipeline(
  task="ner",
  model=trainer.model,
  batch_size=8,
  tokenizer=tokenizer,
  device="cuda",
)

In [ ]:
quick_check = (
    "Bệnh nhân Nguyễn Văn A, 35 tuổi, trú tại Cầu Giấy, Hà Nội. "
    "Nhập viện ngày 20/01/2026 với triệu chứng sốt cao và ho kéo dài."
)
result = tuned_pipeline(quick_check)
result

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'entity': 'B-NAME',
  'score': np.float32(0.67187804),
  'index': 3,
  'word': 'Nguyễn',
  'start': None,
  'end': None},
 {'entity': 'I-NAME',
  'score': np.float32(0.77994615),
  'index': 4,
  'word': 'Văn',
  'start': None,
  'end': None},
 {'entity': 'I-NAME',
  'score': np.float32(0.7398234),
  'index': 5,
  'word': 'A@@',
  'start': None,
  'end': None},
 {'entity': 'B-AGE',
  'score': np.float32(0.7648616),
  'index': 7,
  'word': '35',
  'start': None,
  'end': None},
 {'entity': 'B-LOCATION',
  'score': np.float32(0.95191205),
  'index': 13,
  'word': 'Cầu',
  'start': None,
  'end': None},
 {'entity': 'I-LOCATION',
  'score': np.float32(0.9596495),
  'index': 14,
  'word': 'Giấ@@',
  'start': None,
  'end': None},
 {'entity': 'I-LOCATION',
  'score': np.float32(0.9511133),
  'index': 15,
  'word': 'y@@',
  'start': None,
  'end': None},
 {'entity': 'B-LOCATION',
  'score': np.float32(0.9719957),
  'index': 17,
  'word': 'Hà',
  'start': None,
  'end': None},
 {'entity': 'I-

In [ ]:
model_config = {"batch_size": 8}
signature = mlflow.models.infer_signature(
    [quick_check],
    mlflow.transformers.generate_signature_output(tuned_pipeline, [quick_check]),
    params=model_config,
)

In [ ]:
with mlflow.start_run(run_id=run.info.run_id) :
    model_info = mlflow.transformers.log_model(
        transformers_model=tuned_pipeline,
        name="fine_tuned",
        signature=signature,
        input_example=[quick_check],
        model_config=model_config,
    )

2026/02/24 08:25:40 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/02/24 08:25:40 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/02/24 08:25:53 WARNING mlflow.transformers.model_io: Could not specify device parameter for this pipeline type.Falling back to loading the model with the default device.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026/02/24 08:25:54 WARNING mlflow.transformers: params provided to the `predict` method will override the inference configuration saved with the model. If the params provided are not valid for the pipeline, MlflowException will be raised.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


🏃 View run rogue-bass-551 at: http://34.143.169.73:5000/#/experiments/1/runs/5ca4ee601d234786ae84a93e41384918
🧪 View experiment at: http://34.143.169.73:5000/#/experiments/1
